In [ ]:
ready-to-run code scaffold that implements the full idea in a leakage-safe way:
- ChEMBL scaffold CV (GroupKFold on Murcko scaffolds)
- For each fold: train baseline on ChEMBL-train
- Score COCONUT-A in chunks (so you don’t load 386k fingerprints into RAM)
- Select high-confidence pseudo-negatives (p ≤ ε)
- Enforce diversity via Butina clustering (Tanimoto distance cutoff)
- Add only as many pseudo-negatives as needed to balance the fold
- Retrain with sample weights (pseudo weight < 1)
- Evaluate on ChEMBL test fold only (metrics)

This uses MorganGenerator to avoid your RDKit deprecation spam.

In [10]:
#--------------------------------------------------
#    (0)              Imports 
# --------------------------------------------------
#   
import os
import math
import numpy as np
import pandas as pd

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef, balanced_accuracy_score

# Optional: silence RDKit warnings (kekulize etc.)
# RDLogger.DisableLog("rdApp.warning")

# -------------------------
# PATHS (adjust if needed)
# -------------------------
BASE = r"C:\Users\Besitzer\Desktop\M3_databases"
TRAIN_CSV = os.path.join(BASE, "ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv")
COCO_CSV  = os.path.join(BASE, "coconut_screen_out", "coconut_screen_ranked.csv")  # must contain SMILES
OUTDIR    = os.path.join(BASE, "pseudo_neg_controls_scaffoldcv")
os.makedirs(OUTDIR, exist_ok=True)    

#----------------------------------------------------------------
#    (1)   Helpers: labels, mol parsing, scaffolds, fingerprints 
# ---------------------------------------------------------------
# 
def label_to_binary(consensus_label: str) -> int:
    """
    Map your consensus_label to binary.
    Positives: active, active_single
    Negatives: inactive, inactive_single
    """
    if consensus_label in ("active", "active_single"):
        return 1
    if consensus_label in ("inactive", "inactive_single"):
        return 0
    return None

def safe_mol_from_smiles(smiles: str):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        m = Chem.MolFromSmiles(smiles)
        return m
    except Exception:
        return None

def murcko_scaffold_smiles(mol) -> str:
    """Return Murcko scaffold SMILES or None."""
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None:
            return None
        return Chem.MolToSmiles(scaf, isomericSmiles=False)
    except Exception:
        return None

# MorganGenerator (modern RDKit)
MORGAN_RADIUS = 2
MORGAN_NBITS  = 2048
_morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS)

def mol_to_fp(mol):
    """Return RDKit ExplicitBitVect fingerprint or None."""
    try:
        return _morgan_gen.GetFingerprint(mol)
    except Exception:
        return None
    
#----------------------------------------------------------------
#    (2)   Butina clustering (diversity control)
# ---------------------------------------------------------------
#   
def butina_cluster_fps(fps, cutoff_dist=0.65):
    """
    Cluster fingerprints with Butina.
    cutoff_dist is distance = 1 - similarity.
    Returns list of clusters (each cluster is list of indices).
    """
    if len(fps) == 0:
        return []
    if len(fps) == 1:
        return [[0]]

    # distance matrix in the form expected by Butina: lower triangular list
    dists = []
    for i in range(1, len(fps)):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1.0 - x for x in sims])

    clusters = Butina.ClusterData(dists, len(fps), cutoff_dist, isDistData=True)
    # clusters returned as tuples of indices, largest clusters first
    return [list(c) for c in clusters]



#----------------------------------------------------------------
#    (3)   Read ChEMBL, compute scaffolds + fingerprints
# ---------------------------------------------------------------
#   
def load_chembl_train(train_csv):
    df = pd.read_csv(train_csv)
    df["y"] = df["consensus_label"].map(label_to_binary)
    df = df[df["y"].isin([0, 1])].copy()

    # pick smiles column name (adjust if your file differs)
    smiles_col = None
    for c in ["canonical_smiles", "smiles", "Smiles", "SMILES"]:
        if c in df.columns:
            smiles_col = c
            break
    if smiles_col is None:
        raise ValueError("Could not find a SMILES column in ChEMBL CSV.")

    df["mol"] = df[smiles_col].apply(safe_mol_from_smiles)
    df = df[df["mol"].notnull()].copy()

    df["scaffold"] = df["mol"].apply(murcko_scaffold_smiles)
    df = df[df["scaffold"].notnull()].copy()

    df["fp"] = df["mol"].apply(mol_to_fp)
    df = df[df["fp"].notnull()].copy()

    return df, smiles_col

def fps_to_numpy(fps):
    X = np.zeros((len(fps), MORGAN_NBITS), dtype=np.uint8)
    for i, fp in enumerate(fps):
        arr = np.zeros((MORGAN_NBITS,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        X[i, :] = arr
    return X

#----------------------------------------------------------------
#    (4)   Split COCONUT into A/B by scaffold (streaming-friendly)
# ---------------------------------------------------------------
""" Because COCONUT is big, we do a first pass to assign scaffold and 
    then split scaffolds into A/B.
    Pragmatically: you can also do an initial random split by hash of 
    scaffold (deterministic) without storing all scaffolds in RAM.
    Deterministic scaffold split by hash: 
    This avoids holding millions of scaffold strings."""

import hashlib

def scaffold_to_half(scaffold: str, split_ratio=0.5) -> str:
    """
    Deterministic split: map scaffold string -> A or B.
    """
    h = hashlib.md5(scaffold.encode("utf-8")).hexdigest()
    # use first 8 hex digits as int
    x = int(h[:8], 16) / float(16**8)
    return "A" if x < split_ratio else "B"

#----------------------------------------------------------------
#    (5)   Stream COCONUT-A, score, and collect pseudo-negative candidates
# ---------------------------------------------------------------

""" Key idea: don’t fingerprint everything.
    We only keep candidates with p <= eps, and we cap candidates for clustering."""

def collect_pseudo_neg_candidates(
    coco_csv,
    model,
    eps=0.01,
    max_candidates=20000,
    chunksize=20000,
    cutoff_dist=0.65,
    require_scaffold=True,
    smiles_col_guess=("canonical_smiles", "smiles", "SMILES", "Smiles"),
):
    """
    Stream through COCONUT CSV, keep low-prob candidates from COCONUT-A only.
    Returns list of dicts: [{"smiles":..., "fp":..., "p":..., "scaffold":...}, ...]
    """
    # find smiles column by reading header
    head = pd.read_csv(coco_csv, nrows=5)
    smi_col = None
    for c in smiles_col_guess:
        if c in head.columns:
            smi_col = c
            break
    if smi_col is None:
        raise ValueError("Could not find a SMILES column in COCONUT CSV.")

    kept = []
    worst_p_in_kept = -1.0  # track current worst p among kept (for trimming)

    for chunk in pd.read_csv(coco_csv, chunksize=chunksize):
        if smi_col not in chunk.columns:
            continue

        smiles_list = chunk[smi_col].astype(str).tolist()

        # Build fps for this chunk and predict in mini-batches
        mols = [safe_mol_from_smiles(s) for s in smiles_list]
        scaffolds = [murcko_scaffold_smiles(m) if m is not None else None for m in mols]

        # filter: must have scaffold to be split + to avoid garbage
        idx_ok = [i for i, sc in enumerate(scaffolds) if sc is not None]
        if not idx_ok:
            continue

        # choose only COCONUT-A by scaffold hash
        idx_A = [i for i in idx_ok if scaffold_to_half(scaffolds[i]) == "A"]
        if not idx_A:
            continue

        fps = []
        meta = []
        for i in idx_A:
            fp = mol_to_fp(mols[i])
            if fp is None:
                continue
            fps.append(fp)
            meta.append((smiles_list[i], scaffolds[i]))

        if not fps:
            continue

        X = fps_to_numpy(fps)
        p = model.predict_proba(X)[:, 1]

        for (smi, scaf), pi, fp in zip(meta, p, fps):
            if pi <= eps:
                kept.append({"smiles": smi, "scaffold": scaf, "p": float(pi), "fp": fp})

        # trim to max_candidates by keeping lowest p
        if len(kept) > max_candidates:
            kept.sort(key=lambda d: d["p"])
            kept = kept[:max_candidates]

    # sort final by p
    kept.sort(key=lambda d: d["p"])
    return kept

#----------------------------------------------------------------
#    (6)   Select diverse pseudo-negatives via Butina and sample to balance
# ---------------------------------------------------------------

def select_diverse_pseudo_negs(candidates, n_needed, cutoff_dist=0.65):
    """
    candidates: list of dicts with fp, p
    Returns selected list length <= n_needed
    """
    if n_needed <= 0 or len(candidates) == 0:
        return []

    fps = [d["fp"] for d in candidates]
    clusters = butina_cluster_fps(fps, cutoff_dist=cutoff_dist)

    # representative: pick the lowest-p member of each cluster (best "inactive")
    selected = []
    for cl in clusters:
        cl_sorted = sorted(cl, key=lambda idx: candidates[idx]["p"])
        best_idx = cl_sorted[0]
        selected.append(candidates[best_idx])

    # now take up to n_needed, again by lowest p
    selected.sort(key=lambda d: d["p"])
    return selected[:n_needed]

#----------------------------------------------------------------
#    (7)   Fold-wise scaffold CV with pseudo-negative balancing
# ---------------------------------------------------------------

def train_lr():
    # strong baseline; adjust C if desired
    return LogisticRegression(
        max_iter=4000,
        solver="lbfgs",
        n_jobs=1
    )

def run_scaffoldcv_with_pseudoneg(
    chembl_df,
    eps=0.01,
    butina_cutoff_dist=0.65,
    pseudo_weight=0.3,
    max_coco_candidates=20000
):
    y = chembl_df["y"].values.astype(int)
    scaff = chembl_df["scaffold"].values
    fps = chembl_df["fp"].tolist()
    X = fps_to_numpy(fps)

    gkf = GroupKFold(n_splits=10)

    rows = []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=scaff), start=1):
        X_tr, y_tr = X[tr], y[tr]
        X_te, y_te = X[te], y[te]

        # baseline fit on ChEMBL train fold
        base_model = train_lr()
        base_model.fit(X_tr, y_tr)

        # determine how many pseudo-negs needed to balance THIS fold
        n_pos = int((y_tr == 1).sum())
        n_neg = int((y_tr == 0).sum())
        n_needed = max(0, n_pos - n_neg)

        print(f"[Fold {fold}] train pos={n_pos} neg={n_neg} -> need pseudo_neg={n_needed}")

        # collect pseudo-neg candidates from COCONUT-A scored by fold-model
        candidates = collect_pseudo_neg_candidates(
            COCO_CSV,
            model=base_model,
            eps=eps,
            max_candidates=max_coco_candidates,
            chunksize=20000,
            cutoff_dist=butina_cutoff_dist
        )

        pseudo = select_diverse_pseudo_negs(
            candidates,
            n_needed=n_needed,
            cutoff_dist=butina_cutoff_dist
        )

        print(f"[Fold {fold}] candidates={len(candidates)} | selected pseudo={len(pseudo)}")

        # build augmented training data
        if len(pseudo) > 0:
            X_pseudo = fps_to_numpy([d["fp"] for d in pseudo])
            y_pseudo = np.zeros((len(pseudo),), dtype=int)

            X_aug = np.vstack([X_tr, X_pseudo])
            y_aug = np.concatenate([y_tr, y_pseudo])

            w = np.ones((len(y_aug),), dtype=float)
            w[len(y_tr):] = float(pseudo_weight)  # down-weight pseudo-labels
        else:
            X_aug, y_aug = X_tr, y_tr
            w = None

        # retrain final model for this fold
        model = train_lr()
        if w is None:
            model.fit(X_aug, y_aug)
        else:
            model.fit(X_aug, y_aug, sample_weight=w)

        # evaluate on ChEMBL test fold only
        p_te = model.predict_proba(X_te)[:, 1]
        y_hat = (p_te >= 0.5).astype(int)

        roc = roc_auc_score(y_te, p_te)
        pr  = average_precision_score(y_te, p_te)
        mcc = matthews_corrcoef(y_te, y_hat)
        bacc= balanced_accuracy_score(y_te, y_hat)

        rows.append({
            "fold": fold,
            "n_train_pos": n_pos,
            "n_train_neg": n_neg,
            "n_pseudo": len(pseudo),
            "roc_auc": roc,
            "pr_auc": pr,
            "mcc": mcc,
            "bal_acc": bacc
        })

        print(f"[Fold {fold}] ROC={roc:.3f} PR={pr:.3f} MCC={mcc:.3f} BalAcc={bacc:.3f}")

    res = pd.DataFrame(rows)
    res["model"] = "pseudo_neg"
    res["eps"] = eps
    res["butina_cutoff_dist"] = butina_cutoff_dist
    res["pseudo_weight"] = pseudo_weight

    return res

def run_scaffoldcv_baseline(chembl_df):
    """
    Scaffold-CV on ChEMBL only (no pseudo-negatives).
    Returns per-fold results DataFrame.
    """
    y = chembl_df["y"].values.astype(int)
    scaff = chembl_df["scaffold"].values
    fps = chembl_df["fp"].tolist()
    X = fps_to_numpy(fps)

    gkf = GroupKFold(n_splits=10)

    rows = []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=scaff), start=1):
        X_tr, y_tr = X[tr], y[tr]
        X_te, y_te = X[te], y[te]

        model = train_lr()
        model.fit(X_tr, y_tr)

        p_te = model.predict_proba(X_te)[:, 1]
        y_hat = (p_te >= 0.5).astype(int)

        roc = roc_auc_score(y_te, p_te)
        pr  = average_precision_score(y_te, p_te)
        mcc = matthews_corrcoef(y_te, y_hat)
        bacc= balanced_accuracy_score(y_te, y_hat)

        rows.append({
            "model": "baseline",
            "fold": fold,
            "n_train_pos": int((y_tr == 1).sum()),
            "n_train_neg": int((y_tr == 0).sum()),
            "n_pseudo": 0,
            "pseudo_weight": np.nan,
            "eps": np.nan,
            "butina_cutoff_dist": np.nan,
            "roc_auc": roc,
            "pr_auc": pr,
            "mcc": mcc,
            "bal_acc": bacc
        })

        print(f"[Baseline Fold {fold}] ROC={roc:.3f} PR={pr:.3f} MCC={mcc:.3f} BalAcc={bacc:.3f}")

    return pd.DataFrame(rows)

def summarize_runs(df_all, metrics=("roc_auc","pr_auc","mcc","bal_acc")):
    """
    Return mean±sd summary table grouped by model settings.
    """
    group_cols = ["model", "eps", "butina_cutoff_dist", "pseudo_weight"]
    out = (df_all
           .groupby(group_cols, dropna=False)[list(metrics)]
           .agg(["mean", "std"])
           .reset_index())
    # flatten columns
    out.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in out.columns]
    return out

def run_ablation_suite(chembl_df, eps=0.01, butina_cutoff_dist=0.65, max_coco_candidates=20000):
    """
    Runs:
      A) baseline
      B) pseudo-neg, weight=1.0
      C) pseudo-neg, weight=0.3
    Returns concatenated per-fold DataFrame and a summary DataFrame.
    """
    print("\n=== A) Baseline (ChEMBL only) ===")
    df_base = run_scaffoldcv_baseline(chembl_df)

    print("\n=== B) Pseudo-neg augmentation (weight=1.0) ===")
    df_w1 = run_scaffoldcv_with_pseudoneg(
        chembl_df,
        eps=eps,
        butina_cutoff_dist=butina_cutoff_dist,
        pseudo_weight=1.0,
        max_coco_candidates=max_coco_candidates
    )

    print("\n=== C) Pseudo-neg augmentation (weight=0.3) ===")
    df_w03 = run_scaffoldcv_with_pseudoneg(
        chembl_df,
        eps=eps,
        butina_cutoff_dist=butina_cutoff_dist,
        pseudo_weight=0.3,
        max_coco_candidates=max_coco_candidates
    )

    df_all = pd.concat([df_base, df_w1, df_w03], ignore_index=True)
    df_sum = summarize_runs(df_all)

    return df_all, df_sum

#----------------------------------------------------------------
#    (8)   Run everything and save results
# ---------------------------------------------------------------

def main():
    chembl_df, chembl_smiles_col = load_chembl_train(TRAIN_CSV)

    print("Loaded:", chembl_df.shape)
    print("Label counts:\n", chembl_df["consensus_label"].value_counts())

    df_all, df_sum = run_ablation_suite(
        chembl_df,
        eps=0.01,
        butina_cutoff_dist=0.65,
        max_coco_candidates=20000
    )

    out_all = os.path.join(OUTDIR, "scaffoldcv_ablation_perfold.csv")
    out_sum = os.path.join(OUTDIR, "scaffoldcv_ablation_summary.csv")

    df_all.to_csv(out_all, index=False)
    df_sum.to_csv(out_sum, index=False)

    print("\nSaved:")
    print(" -", out_all)
    print(" -", out_sum)

    print("\nAblation summary (mean±sd):")
    print(df_sum)

    print("\nFold counts:")
    print(df_all.groupby(["model", "pseudo_weight"])["fold"].nunique())
    print("\nFold values per condition:")
    print("baseline folds:", sorted(df_all[df_all["model"]=="baseline"]["fold"].unique()))
    print("pseudo w=1 folds:", sorted(df_all[(df_all["model"]=="pseudo_neg") & (df_all["pseudo_weight"]==1.0)]["fold"].unique()))


    from scipy.stats import ttest_rel, wilcoxon

    baseline = df_all[df_all["model"] == "baseline"][["fold", "mcc", "bal_acc", "roc_auc"]]
    pseudo_w1 = df_all[(df_all["model"] == "pseudo_neg") & (df_all["pseudo_weight"] == 1.0)][["fold", "mcc", "bal_acc", "roc_auc"]]

    paired = baseline.merge(pseudo_w1, on="fold", suffixes=("_base", "_pseudo"))
    print("\nPaired folds used:", paired["fold"].tolist())
    print("n paired folds:", len(paired))

    for metric in ["mcc", "bal_acc", "roc_auc"]:
        t = ttest_rel(paired[f"{metric}_base"], paired[f"{metric}_pseudo"])
        print(f"{metric}: paired t-test p = {t.pvalue:.4f} (t={t.statistic:.3f}, n={len(paired)})")
        try:
            w = wilcoxon(paired[f"{metric}_base"], paired[f"{metric}_pseudo"])
            print(f"{metric}: Wilcoxon p = {w.pvalue:.4f}")
        except Exception as e:
            print(f"{metric}: Wilcoxon failed: {e}")

    diff = (paired["mcc_pseudo"] - paired["mcc_base"]).astype(float)
    diff = diff.replace([np.inf, -np.inf], np.nan).dropna()

    sd = diff.std(ddof=1)

    print("diff values:", diff.values)
    print("diff mean:", float(diff.mean()))
    print("diff sd:", float(sd))

    if len(diff) < 2:
        print("Cohen's d (paired): not enough paired folds")
    elif sd == 0 or np.isnan(sd):
        print("Cohen's d (paired): undefined (sd=0 or NaN)")
    else:
        d = float(diff.mean() / sd)
        print(f"Cohen's d (paired): {d:.3f}")  

    # 95% CI for mean delta (paired)
    from scipy.stats import t
    n = len(diff)
    se = diff.std(ddof=1) / np.sqrt(n)
    ci = t.interval(0.95, df=n-1, loc=diff.mean(), scale=se)
    print(f"Mean ΔMCC = {diff.mean():+.4f} (95% CI {ci[0]:+.4f} to {ci[1]:+.4f})")
    
    return df_all, df_sum

# Notebook run
df_all, df_sum = main()


Loaded: (2268, 17)
Label counts:
 consensus_label
active_single      1502
inactive_single     463
active              286
inactive             17
Name: count, dtype: int64

=== A) Baseline (ChEMBL only) ===
[Baseline Fold 1] ROC=0.997 PR=0.999 MCC=0.883 BalAcc=0.925
[Baseline Fold 2] ROC=0.986 PR=0.997 MCC=0.845 BalAcc=0.923
[Baseline Fold 3] ROC=0.996 PR=0.999 MCC=0.875 BalAcc=0.924
[Baseline Fold 4] ROC=0.981 PR=0.997 MCC=0.766 BalAcc=0.826
[Baseline Fold 5] ROC=0.982 PR=0.996 MCC=0.821 BalAcc=0.925
[Baseline Fold 6] ROC=0.988 PR=0.994 MCC=0.872 BalAcc=0.929
[Baseline Fold 7] ROC=0.983 PR=0.990 MCC=0.851 BalAcc=0.917
[Baseline Fold 8] ROC=0.986 PR=0.997 MCC=0.798 BalAcc=0.931
[Baseline Fold 9] ROC=0.954 PR=0.992 MCC=0.626 BalAcc=0.859
[Baseline Fold 10] ROC=0.972 PR=0.987 MCC=0.705 BalAcc=0.804

=== B) Pseudo-neg augmentation (weight=1.0) ===
[Fold 1] train pos=1604 neg=437 -> need pseudo_neg=1167


[19:47:30] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[19:47:30] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[Fold 1] candidates=13605 | selected pseudo=1167
[Fold 1] ROC=0.998 PR=0.999 MCC=0.898 BalAcc=0.936
[Fold 2] train pos=1600 neg=441 -> need pseudo_neg=1159


[19:49:56] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[19:49:56] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[Fold 2] candidates=17481 | selected pseudo=1159
[Fold 2] ROC=0.986 PR=0.997 MCC=0.845 BalAcc=0.923
[Fold 3] train pos=1589 neg=452 -> need pseudo_neg=1137


...

[Fold 8] candidates=14429 | selected pseudo=1155
[Fold 8] ROC=0.986 PR=0.997 MCC=0.798 BalAcc=0.931
[Fold 9] train pos=1597 neg=445 -> need pseudo_neg=1152


[20:31:15] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[20:31:15] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[Fold 9] candidates=19811 | selected pseudo=1152
[Fold 9] ROC=0.953 PR=0.991 MCC=0.626 BalAcc=0.859
[Fold 10] train pos=1628 neg=414 -> need pseudo_neg=1214


[20:33:59] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[20:33:59] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[Fold 10] candidates=14601 | selected pseudo=1214
[Fold 10] ROC=0.972 PR=0.987 MCC=0.739 BalAcc=0.827

Saved:
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_ablation_perfold.csv
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_ablation_summary.csv

Ablation summary (mean±sd):
        model   eps  butina_cutoff_dist  pseudo_weight  roc_auc_mean  \
0    baseline   NaN                 NaN            NaN      0.982474   
1  pseudo_neg  0.01                0.65            0.3      0.982690   
2  pseudo_neg  0.01                0.65            1.0      0.982872   

   roc_auc_std  pr_auc_mean  pr_auc_std  mcc_mean   mcc_std  bal_acc_mean  \
0     0.012485     0.994847    0.004174  0.804190  0.083548      0.896095   
1     0.012783     0.994921    0.004206  0.813953  0.081568      0.902047   
2     0.012720     0.994970    0.004178  0.816708  0.077110      0.902808   

   bal_acc_std  
0     0.047754  
1     0.042147  
2 